In [ ]:
!pip install -U -q transformers sentence-transformers chromadb accelerate

In [ ]:
import json
import chromadb

from sentence_transformers import SentenceTransformer

#Carregar Notícias

In [ ]:
with open("noticias_2.json", "r", encoding="utf-8") as f:
    noticias = json.load(f)

print(f"{len(noticias)} notícias carregadas.")

#Carregar modelo de embedding

In [ ]:
modelo_embedding = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")
print("Modelo de embeddings carregado.")

#Preparar documentos

In [ ]:
documentos = []
ids = []
metadados = []

for i, noticia in enumerate(noticias):

    documento = (
        f"{noticia['titulo']}\n"
        f"{noticia['resumo']}"
    )

    documentos.append(documento)

    ids.append(f"{noticia['ativo'].lower()}_{i}")

    metadados.append({
        "ativo": noticia["ativo"],
        "data": noticia["data"],
        "fonte": noticia["fonte"]
    })

#Gerar embeddings

In [ ]:
embeddings = modelo_embedding.encode(
    documentos,
    normalize_embeddings=True
).tolist()

print(f"{len(embeddings)} embeddings gerados.")

#Criar banco vetorial

In [ ]:
chroma_client = chromadb.Client()

try:
    chroma_client.delete_collection(
        name="noticias_carteira"
    )
except Exception:
    pass

colecao_noticias = chroma_client.create_collection(
    name="noticias_carteira",
    metadata={"hnsw:space": "cosine"}
)

#Indexar no chromaDB

In [ ]:
colecao_noticias.add(
    embeddings=embeddings,
    documents=documentos,
    ids=ids,
    metadatas=metadados
)

print(f"{len(documentos)} notícias indexadas em 'noticias_carteira'.")

#Teste de Recuperação

In [ ]:
consulta = "dividendos recentes"

vetor_consulta = modelo_embedding.encode(
    [consulta],
    prompt_name="query",
    normalize_embeddings=True
).tolist()


for ativo in ["PETR4", "MXRF11", "CMIG4"]:

    resultado = colecao_noticias.query(
        query_embeddings=vetor_consulta,
        n_results=2,
        where={"ativo": ativo},
        include=["documents", "distances"]
    )

    print(f"\n[{ativo}]")

    for documento, distancia in zip(
        resultado["documents"][0],
        resultado["distances"][0]
    ):
        print(f"[dist={distancia:.3f}] {documento}")